# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [ ]:
# imports
import os
from openai import OpenAI
from dotenv import load_dotenv

from IPython.display import Markdown, display, update_display

In [2]:
# constants
MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [ ]:
# set up environment
load_dotenv(override=True)

api_key = os.getenv(f"OPENAI_API_KEY")

if api_key and len(api_key) > 10 and api_key.startswith('sk-'):
    print("API key looks good")
else:
    print("Something did not work")


base_url = os.getenv("BASE_URL")
base_url_local = os.getenv("BASE_URL_LOCAL")

API key looks good


In [19]:
# here is the question; type over this to ask something new
system_prompt = """
You are an helpful assistant, providing valuable insights and explanation to the user questions.
"""

user_prompt_prefix = """
Please help me elaboreate on the following question / topic:
"""

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""


messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt_prefix + question}
    ]

In [ ]:
# Get gpt-4o-mini to answer, with streaming
openai = OpenAI(
    base_url=base_url,
    api_key=api_key
)

response = openai.chat.completions.create(
    model = MODEL_GPT,
    messages = messages
)

response = response.choices[0].message.content
display(Markdown(response))

ChatCompletion(id='gen-1762696074-MapPSIOAa7TP1c0fyf0h', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The code snippet you\'ve provided uses Python\'s `yield from` expression in conjunction with a set comprehension. Let\'s break this down step by step to understand what it does and why it might be used.\n\n### Breakdown of the Code\n\n1. **Set Comprehension**:\n   ```python\n   {book.get("author") for book in books if book.get("author")}\n   ```\n   This part creates a set of authors from a collection of `books`. \n\n   - `books`: This variable is expected to be an iterable (likely a list or similar) containing dictionaries, where each dictionary represents a book.\n   - `book.get("author")`: This method attempts to retrieve the value associated with the key "author" from each book\'s dictionary. If the key does not exist or its value is `None`, it will return `None`.\n   - `if book.get("author")`: This conditional filters out any

In [ ]:
# streaming version with GPT
openai = OpenAI(
    base_url = base_url,
    api_key = api_key
)

stream = openai.chat.completions.create(
    model=MODEL_GPT,
    messages=messages,
    stream=True
)

response = ""
display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
    response += chunk.choices[0].delta.content
    update_display(Markdown(response), display_id=display_handle.display_id)

The code snippet you've provided is a combination of a generator expression and a set comprehension, utilizing Python’s `yield from` statement. Let’s break it down step-by-step:

### Components of the Code:

1. **Set Comprehension**: 
   ```python
   {book.get("author") for book in books if book.get("author")}
   ```
   - This part of the code creates a set of unique authors from a collection of `books`.
   - It iterates over an iterable `books` (which should be a list or any iterable collection containing book data).
   - For each `book`, it attempts to retrieve the value associated with the key `"author"` using the `get` method.
   - `if book.get("author")` ensures that only non-`None` (or truthy) values for authors are included in the set. This prevents adding any `None` or empty author entry into the set.

2. **`yield from` Statement**:
   ```python
   yield from ...
   ```
   - The `yield from` expression is used to yield all values from a generator or iterable. This means that instead of returning a single value, it effectively "delegates" the yielding to another iterable.
   - It allows you to yield all the elements from the set comprehension directly to the caller of the generator that contains this expression.

### Purpose of the Code:

- **Unique Authors**: The overall purpose of this code is to extract and yield unique authors from a list of books. This is useful when you want to get a distinct list of authors without duplicates. By utilizing a set comprehension, you take advantage of the property of sets that inherently eliminates duplicate entries.
  
- **Efficient Yielding**: Using `yield from` allows the function (that this code might reside within) to yield each unique author one at a time. This is particularly useful for memory efficiency, especially when dealing with a large number of books, as it does not create a complete list or set in memory upfront, but instead yields each item on demand.

### Example Scenario:

Suppose you have a list of books structured as dictionaries, like so:

```python
books = [
    {"title": "Book 1", "author": "Author A"},
    {"title": "Book 2", "author": "Author B"},
    {"title": "Book 3", "author": "Author A"},
    {"title": "Book 4", "author": None},
    {"title": "Book 5", "author": "Author C"}
]
```

If you run the code:

```python
for author in (yield from {book.get("author") for book in books if book.get("author")}):
    print(author)
```

The output would be:

```
Author A
Author B
Author C
```

### Conclusion:

This line of code provides an efficient way to extract all unique authors from a collection of books without any duplicates or empty entries. The use of `yield from` ensures that these unique authors can be processed lazily, which can be particularly useful for larger datasets.

In [18]:
import requests

requests.get("http://localhost:11434").content

b'Ollama is running'

In [ ]:
# Get Llama 3.2 to answer
ollama = OpenAI(
    base_url=base_url_local,
    api_key='random'
)

response = ollama.chat.completions.create(
    model = MODEL_LLAMA,
    messages = messages
)

response = response.choices[0].message.content
display(Markdown(response))

Let's break down this line of code and understand its functionality.

**Code Explanation:**

`yield from {book.get("author") for book in books if book.get("author")}` is a nested `for` loop expression in Python. It uses the `yield from` syntax introduced in Python 3.3.

Here's what it does:

1. **Nested generator (loop)**: The expression creates a new generator that yields values from an inner iterable (`{book.get("author") for book in books if book.get("author")}`).
2. **Inner iterable**: This is another generator expression (`{book.get("author") for book in books if book.get("author")}`). It:
	* Iterates over the `books` collection.
	* Filters out books that do not have a matching `"author"` value (using the `if` condition).
	* For each valid book, it yields its corresponding author's name using the `get()` method (`book.get("author")`).

**What does `yield from` do?**

The `yield from` keyword is used to delegate an iteration to a sub-iterator. In this case:

1. When the outer generator receives an iteration request (e.g., to retrieve the next author's name), it delegates the iteration to its inner sub-iterator.
2. The inner sub-iterator yields values by iterating over its own iterator (in this case, the filtered books and their authors).
3. The `yield from` keyword "unwraps" the values yielded by the inner sub-iterator and makes them available in the outer iterable.

**Example Use Case:**

Suppose you have a list of dictionaries (`books`) where each dictionary represents a book with an `"author"` field:

```python
books = [
    {"title": "Book 1", "author": "Author A"},
    {"title": "Book 2", "author": "Author B"},
    {"title": "Book 3"},  # Without author data
    {"title": "Book 4", "author": "Author D"}
]

# Using the nested generator expression:
for author in yield from {book.get("author") for book in books if book.get("author")}:
    print(author)
```

This would output:

* `Author A`
* `Author B`
* `Author D` (Note that Book 3's author data is excluded due to the filter condition).

By using this expression, you can easily retrieve all authors from your list of books in a concise and readable way.

In [ ]:
# streaming response using Ollama

ollama = OpenAI(
    base_url=base_url_local,
    api_key='default'
)

stream = ollama.chat.completions.create(
    model=MODEL_LLAMA,
    messages=messages,
    stream=True
)

response = ""
display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
    response += chunk.choices[0].delta.content
    update_display(Markdown(response), display_id=display_handle.display_id)

Let's break down this line of code.

**Understanding the context:**

This code is written in Python and appears to be part of a data processing or iteration task, possibly related to working with JSON or XML data structures (`books`).

**Code analysis:**

The line `yield from {book.get("author") for book in books if book.get("author")}` can be divided into three main parts:

1. `{book.get("author") for book in books if book.get("author")}`:
   - This is a generator expression.
   - It iterates over the `books` data structure (`books`).
   - For each book, it attempts to get the value of the `"author"` key using `.get()`.
   - If the key exists and its value is truthy (i.e., not explicitly set to `None`), it includes the author's name in the iteration.

2. `yield from`:
   - This is a Python 3.3+ syntax used for yielding values from a sub-generator.
   - When called as part of a function definition (`def gen(): yield ...:'), it makes the generator expression within it a top-level generator, allowing it to be used recursively in calls like `gen2 = yield (from gen()).` 
     or simply yielded by its recursive call.

3. `yield from {expression}`:
   - When `yield from` is used with an expression that is already iterable, this statement will then generate values from each individual iteration in the contained generator, rather than the entire container itself.
   
In essence, this line of code yields each author name found in the books data structure. However, it specifically **only** includes authors for which the `"author"` key exists and has a non-`None` value associated with it.

Here's how you can illustrate its use:

```python
books = [
    {"id": 1, "title": "Book A", "author": "AuthorX"},
    {"id": 2, "title": "Book B", "", },
    {"id": 3, 'title':'Book C',"author":"AuthorY"},
]

def gen_authors():
    return {book.get("author") for book in books if book.get("author")}
    
# Create a generator using the expression
gen = gen_authors()

for author in gen:
    print(author)

for yep in gen_authors():
    print(yep)
```

When you run `print(yep)`, yep would also get the value of `"AuthorX"`